# Your Agent May Be Tuned For A Game That No Longer Exists

`kaggle-environments` **1.32.6** changed Kaggriculture's demand model mid-competition. If
you tuned crop targets, reserve prices, or acreage before that, some of your numbers now
describe an economy the competition is no longer running.

I found this the hard way: two days of tuning against 1.32.5, and my best agent's win rate
against a fixed opponent went from **43.8% to 0%** the moment I upgraded. Nothing about my
agent changed. The game did.

This notebook is the check I wish I'd run earlier, plus the quantified diff. Everything is
computed from the engine you have installed, so you can re-run it and see your own numbers.

One thing I only learned by publishing this: **the default Kaggle notebook image ships
`kaggle-environments` 1.29.3** — seven releases behind the 1.32.6 your submission is scored
against. Not one version behind; seven. The first version of this notebook crashed on Kaggle
for exactly that reason, which is how I found out. So if you have been developing in a fresh
Kaggle notebook without an explicit upgrade, you have been measuring a substantially older
game. The first cells detect this and fix it.

*(High-school student doing this competition as an economics project. Corrections very
welcome in the comments — I'd rather be right than upvoted.)*

## 1. The 30-second check

If this prints `STALE`, some of your tuning may be measuring the old economy.

In [ ]:
# Read the SHIPPED version WITHOUT importing the package, then upgrade before any import.
# (Upgrading in-process and calling importlib.reload on kaggle_environments does not work:
#  it has many submodules and the reload raises ImportError partway through. Ask me how I
#  know -- that was version 2 of this notebook.)
import importlib.metadata as md_
shipped = md_.version("kaggle-environments")
print("kaggle-environments as shipped by this image:", shipped)

In [ ]:
!pip install -q -U kaggle-environments 2>/dev/null | tail -1

**If that printed something below 1.32.6 — that is the finding, not a mistake.**

When I ran this on Kaggle it reported **1.29.3**. The engine your submission is actually
scored against is 1.32.6. That is seven releases of drift sitting silently under every
local measurement made in a default notebook, and it is why the first version of this
notebook crashed here: it referenced a constant that does not exist in 1.29.3.

Upgrading is one line, and it has to happen before the first import:

In [ ]:
from kaggle_environments.envs.kaggriculture import kaggriculture as K

now = md_.version("kaggle-environments")
old = hasattr(K, "TOWN_CENTER_DEMAND_SCHEDULE")   # removed in 1.32.6
new = hasattr(K, "MAX_SHOP_INSTANCES")            # added in 1.32.6

print(f"shipped: {shipped}     after upgrade: {now}")
print(f"TOWN_CENTER_DEMAND_SCHEDULE present : {old}   (removed in 1.32.6)")
print(f"MAX_SHOP_INSTANCES present          : {new}   (added in 1.32.6)")
print()
print("VERDICT:", "STALE - pre-rebalance engine" if old else "CURRENT - post-rebalance engine")
if shipped != now:
    print(f"NOTE: this image shipped {shipped}. If you had not upgraded, every number")
    print("      below would describe the pre-rebalance economy.")

Two things worth knowing about that first flag:

- `TOWN_CENTER_DEMAND_SCHEDULE` was **deleted**, so any code that imports it by name now
  raises `ImportError` — a loud failure, which is the good case.
- The quiet failure is worse: if you *copied its values* into your own agent (a lot of us
  mirror the engine constants so the agent can price things without importing the engine),
  nothing breaks. Your agent just prices a demand curve that is no longer there.

## 2. What actually changed

Three edits, all in the town-demand path:

| | 1.32.5 | 1.32.6 |
|---|---|---|
| `townCenterSellInterval` default | 12 turns (2 ticks/day) | **24 turns (1 tick/day)** |
| town-centre multiplier | `1x` → `2x` from day 10 → `4x` from day 20 | **flat `1x`, schedule deleted** |
| shop unlocks | 8 distinct, sampled *without* replacement | **sampled *with* replacement**, capped at `MAX_SHOP_INSTANCES = 8` |

The first two compound. The town centre used to consume every product twice a day, rising
to eight times a day in the last third of the season. Now it consumes each product **once
per day, all season**.

In [ ]:
TURNS_PER_DAY, DAYS = 24, 30

# --- the OLD (1.32.5) values, recorded here for comparison. Not importable any more. ---
OLD_CENTER_INTERVAL = 12                      # -> 2 ticks/day
OLD_CENTER_SCHEDULE = [(20, 4), (10, 2), (0, 1)]

old_centre_season = sum(
    (TURNS_PER_DAY // OLD_CENTER_INTERVAL) * next(m for thr, m in OLD_CENTER_SCHEDULE if d >= thr)
    for d in range(DAYS)
)

# --- the CURRENT values, read from your installed engine ---
new_centre_per_day = TURNS_PER_DAY // 24      # townCenterSellInterval default is now 24
new_centre_season  = new_centre_per_day * DAYS

print(f"town centre, per product, per SEASON:  {old_centre_season}  ->  {new_centre_season}"
      f"   ({new_centre_season/old_centre_season - 1:+.0%})")

## 3. The part that changes strategy: per-product demand

Shops are the other half. Because they are now drawn **with replacement**, you can no
longer assume all eight distinct shops appear — you get 8 draws from 8 shops, so the
*expected* number of distinct shops is about 5.25, and duplicates consume independently.

Below is the expected daily sink per product, old versus new.

In [ ]:
SHOP_SELL_INTERVAL = 4                        # 6 ticks/day
ticks_per_day = TURNS_PER_DAY // SHOP_SELL_INTERVAL
n_shops = len(K.SHOPS)

old_shop, new_shop = {}, {}
for p in K.PRODUCTS:
    old_shop[p] = sum(ticks_per_day * (2 if len(pr) == 1 else 1)
                      for pr in K.SHOPS.values() if p in pr)
    # with replacement: each of MAX_SHOP_INSTANCES draws is this shop with prob 1/n_shops
    max_inst = getattr(K, "MAX_SHOP_INSTANCES", n_shops)
    new_shop[p] = sum(max_inst * (1 / n_shops) * ticks_per_day * (2 if len(pr) == 1 else 1)
                      for pr in K.SHOPS.values() if p in pr)

old_centre_day = 2 * 4                        # late-season worst case under the old 4x
rows = []
for p in K.PRODUCTS:
    if p not in K.TOWN_CENTER_PRODUCTS and new_shop[p] == 0:
        old_t, new_t = old_shop[p], new_shop[p]           # fertilizer: no demand either way
    else:
        old_t = old_shop[p] + (old_centre_day if p in K.TOWN_CENTER_PRODUCTS else 0)
        new_t = new_shop[p] + (1 if p in K.TOWN_CENTER_PRODUCTS else 0)
    rows.append((p, old_t, new_t))

print(f"{'product':<12}{'old sink/day':>14}{'new sink/day':>14}{'change':>10}")
print("-" * 50)
for p, o, n in sorted(rows, key=lambda r: (r[2] - r[1]) / max(r[1], 1)):
    ch = f"{n/o - 1:+.0%}" if o else "  n/a"
    print(f"{p:<12}{o:>14.0f}{n:>14.1f}{ch:>10}")

### Melon is the headline

Melon has **no shop that demands it** — it never did. Its only buyer was the town centre,
which used to eat up to 8/day late in the season. Now it eats **1/day**.

So melon's total demand across a whole season fell by roughly 88%, and it already had the
harshest glut curve in the game (`above_func = "sq"`, `above_target = 3.60`). If your
acreage plan was fitted before the rebalance, melon is the first number to re-check.

In [ ]:
# How far can you sell each product before the price collapses? (unchanged by the
# rebalance -- the CURVE is the same; what changed is how fast the town drains it back)
def avg_realised(item, qty):
    inv, total = K.MARKET_I0, 0
    for _ in range(qty):
        px = K.market_price(item, inv)
        total += px
        if px > 1:                 # sales at the $1 floor do not add to inventory
            inv += 1
    return total / qty

print(f"{'product':<12}" + "".join(f"{f'Q={q}':>9}" for q in (25, 50, 100, 200, 400)))
print("-" * 58)
for p in ["MELON", "WOOL", "MILK", "STRAWBERRY", "TOMATO", "CARROT", "WHEAT", "EGG"]:
    print(f"{p:<12}" + "".join(f"{avg_realised(p, q):>9.0f}" for q in (25, 50, 100, 200, 400)))
print()
print("Average $/unit realised when YOU sell Q units into your own price impact.")
print("Pair this with the sink table above: the sink is how fast the town undoes it.")

## 4. What I'd re-check if I were you

In the order that cost me the most:

1. **Any absolute price threshold.** If you sell above a fixed dollar reserve, those
   numbers were calibrated against a higher price level. Mine were, and the agent started
   refusing sales it should have made, then dumping them in the endgame liquidation
   instead. Scaling every reserve down recovered a chunk of it.
2. **Melon acreage.** Biggest single change in the table.
3. **Anything fitted by A/B before ~2026-08-07.** Not necessarily wrong — but it was
   measured on a different game, so it is untested rather than validated.
4. **Seed-paired A/B generally.** Worth knowing: weed spawn and shop draw share one RNG
   stream, so two variants that differ in end-of-day tile occupancy will also see
   different shop unlocks. "Same seed" is not as clean a control as it looks. I have not
   independently verified this one — it comes from the competition discussions — but it
   matches noise I could not otherwise explain.

## Sources

The mechanics here are read straight from the installed engine. The rebalance itself and
the runtime limits were confirmed by the hosts in the competition discussion forum. I am
not claiming any of the engine facts as a discovery — they are in the source, and several
community members had them first. What I have added is the quantified before/after and the
staleness check.

If you spot an error, please say so and I'll correct it with credit.